### How the data will flow in the project 

Load dataset

        ↓
Explore the data

        ↓

Train/Validation/Test split

        ↓

Build vocabulary (using ONLY the training set)

        ↓

Tokenize the text

        ↓

Convert tokens to integer IDs

        ↓

Pad/Truncate sequences

        ↓

Convert to PyTorch tensors

        ↓

Create a custom Dataset

        ↓

Create a DataLoader

        ↓

Build the LSTM model

        ↓

Train

        ↓

Evaluate

In [ ]:
from datasets import load_dataset
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [ ]:
data = load_dataset("fancyzhx/ag_news")

In [ ]:
# load_dataset returns a datasetDict which containes multiple splits such as: train and test, and it does not have
# .to_pandas() methods. So, you have to convert each split alone. 
print(data)
train_dataset = data["train"]
test_dataset = data["test"]
print(train_dataset[0])
print(test_dataset[0])

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})
{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}
{'text': "Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.", 'label': 2}


### Data Preprocessing 

In [ ]:
# Convert the train split to pandas
train_df = data['train'].to_pandas()
test_df = data['test'].to_pandas()

In [ ]:
print(train_df.shape)
print("===========")
print(test_df.shape)

(120000, 2)
(7600, 2)


In [ ]:
print(train_df.head())
print(test_df.head())

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2
                                                text  label
0  Fears for T N pension after talks Unions repre...      2
1  The Race is On: Second Private Team Sets Launc...      3
2  Ky. Company Wins Grant to Study Peptides (AP) ...      3
3  Prediction Unit Helps Forecast Wildfires (AP) ...      3
4  Calif. Aims to Limit Farm-Related Smog (AP) AP...      3


In [ ]:
train_df.isnull().sum()

text     0
label    0
dtype: int64

In [ ]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 120000 entries, 0 to 119999
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   text    120000 non-null  str  
 1   label   120000 non-null  int64
dtypes: int64(1), str(1)
memory usage: 28.9 MB


In [ ]:
# Training 
X_train = np.array(train_df.iloc[:, 0]) # Training text
print(X_train)
y_train = np.array(train_df.iloc[:, 1]) # Training label 
print(y_train[0])

# Testing 
X_test = np.array(test_df.iloc[:, 0]) # Testing text
print(X_test[0])
y_test = np.array(test_df.iloc[:, 1]) # Testing label
print(y_test[0])

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
2
Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
2


In [ ]:
# The data is already splitted into training and testing.
# But we need to split the testing dataset into validation data and testing data.

# Split the test data into testing and validation
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5)

#### LSTM  cannot read English, it only understand numbers, so we will have to turn the words into numbers.

#### Step 1. Building the Vocabulary

- Vocabulary: simply a dictionary of all the words your model knows.
1. You read all the training sentences once. 
2. You write down every unique word.
3. Assign each word a number.

#### Step 2. Tokenization

- A token is simply one unit of text, usually a token is one word. 
- The process of splitting text into tokens is called **tokenization**.
- There are different types of tokenization, like: word tokenization or character tokenization.
- In **Step 1** we have the words as individuals, and the vocabulary dictionary is words not sentences, so we must first split the sentences into words i.e. tokenization.
- PyTorch can not convert raw text into tensors, it should be converted into nubmers first, then tokenized.

#### Step 3. Convert Tokens into IDs

- Now we combine the first two steps. Replace every word using the vocabulary. Now a sentence becomes like that:
[0,1,2,3]

- So we will use the dictionary (translation table) to replace each word with its ID.


#### What about words we have never seen them before:

##### For example: Tesla.

- It does not exits in the vocabulary. 
- We create a special token called UNK --> Unknown word. 
- We give it a unique number (ID).

#### Step 4. Padding 

- When training, every sentence has a different length. For example: [3,5,8] and [6,10,8,9,2,0,1].
- A batch simply contain multiple training examples the model processes together (sentences in your case). 
- Neural networks process data in batches, this is way much faster for the model.
- PyTorch stores the batch in a single tenosr. 
- A tensor requires every row to have the same length (rectangular tensors). 
- Since sentences naturally have different lengths they will not be rectangular tensors like this:
- [
  
    [3, 5, 8],

    [6, 10, 2, 8, 1, 3, 9]

]

- Padding fixes this:
    1. We choose a maximum length.
    2. Add a special value called PAD until it reaches length 7 for sentences that are less then 7.
    3. Usually PAD = 0.
    4. When both sentences are have the same length, Now PyTorch can make one tensor. 
    5. Now the **DataLoader** can give this whole batch to the LSTM at once.
- If the sentence is too long we cut it down using **Truncation**.

### Neural Network 

In [ ]:
class AGNewsDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long).to(device) # torch.long because its text not numbers
        self.y = torch.tensor(y, dtype=torch.long).to(device)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, index):
        return self.X[index], self.y[index]

In [ ]:
training_data = AGNewsDataset(X_train, y_train)
validation_data = AGNewsDataset(X_val, y_val)
testing_data = AGNewsDataset(X_test, y_test)

TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.